In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install transformers[sentencepiece] datasets evaluate rouge_score nltk tqdm matplotlib -q


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00


In [5]:
from transformers import pipeline, set_seed
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd
import evaluate
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch

# Download NLTK tokenizer data (only first time)
nltk.download("punkt")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [8]:
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)


pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [9]:
from datasets import load_dataset

dataset_samsum = load_dataset(
    "csv",
    data_files={
        "train": "/content/drive/MyDrive/Colab_Notebooks/samsum/samsum-train.csv",
        "validation": "/content/drive/MyDrive/Colab_Notebooks/samsum/samsum-validation.csv",
        "test": "/content/drive/MyDrive/Colab_Notebooks/samsum/samsum-test.csv"
    }
)

print(dataset_samsum)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


In [10]:
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]
split_lengths


[14732, 818, 819]

In [11]:
print(f"Features: {dataset_samsum['train'].column_names}")


Features: ['id', 'dialogue', 'summary']


In [12]:
print("\nDialogue:")
print(dataset_samsum["test"][1]["dialogue"])

print("\nSummary:")
print(dataset_samsum["test"][1]["summary"])



Dialogue:
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

Summary:
Eric and Rob are going to watch a stand-up on youtube.


In [13]:
dialogue = dataset_samsum['test'][0]['dialogue']
print(dialogue)


Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye


In [14]:
pipe = pipeline('summarization', model=model_ckpt)


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


In [15]:
pipe_out = pipe(dialogue)
pipe_out

Your max_length is set to 128, but your input_length is only 122. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)


[{'summary_text': "Amanda: Ask Larry Amanda: He called her last time we were at the park together .<n>Hannah: I'd rather you texted him .<n>Amanda: Just text him ."}]

In [16]:
print(pipe_out[0]['summary_text'].replace(".<n>",".\n"))

Amanda: Ask Larry Amanda: He called her last time we were at the park together .
Hannah: I'd rather you texted him .
Amanda: Just text him .


In [17]:
import torch
from tqdm import tqdm

def generate_batch_sized_chunks(list_of_elements, batch_size):
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                batch_size=4,                 # smaller batch
                                column_text="dialogue",       # <- set to your columns
                                column_summary="summary",
                                max_src_len=384,              # shorter input than 1024
                                max_new_tokens=64,            # shorter output
                                num_beams=1,                  # beam search off (saves mem)
                                use_fp16=True):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    model.generation_config.use_cache = False   # reduce KV cache memory

    # Optional: switch model weights to fp16 on GPU
    if use_fp16 and device == "cuda":
        try:
            model.half()
        except Exception:
            pass

    torch.cuda.empty_cache() if device == "cuda" else None

    art_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    tgt_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for art_batch, tgt_batch in tqdm(zip(art_batches, tgt_batches), total=len(art_batches)):
        with torch.no_grad():
            # pad to longest in batch (NOT max_length=1024) to save memory
            enc = tokenizer(
                art_batch,
                truncation=True,
                max_length=max_src_len,
                padding=True,             # dynamic padding
                return_tensors="pt",
            ).to(device)

            # FP16 autocast (safe on most seq2seq models)
            autocast_ctx = torch.cuda.amp.autocast if (use_fp16 and device == "cuda") else torch.cpu.amp.autocast
            # torch.cpu.amp.autocast is a no-op on CPU; this keeps code simple

            with (autocast_ctx() if device == "cuda" and use_fp16 else torch.no_grad()):
                summaries = model.generate(
                    **enc,
                    max_new_tokens=max_new_tokens,
                    num_beams=num_beams,
                    do_sample=False,
                    length_penalty=0.8,
                    early_stopping=True,
                )

        decoded = [tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
                   for s in summaries]
        decoded = [d.replace("", " ") for d in decoded]  # keep your original cleanup

        metric.add_batch(predictions=decoded, references=tgt_batch)

        # free intermediate tensors
        del enc, summaries
        if device == "cuda":
            torch.cuda.empty_cache()

    return metric.compute()


In [18]:
import evaluate
rouge = evaluate.load("rouge")

score = calculate_metric_on_test_ds(
    dataset_samsum["test"],
    rouge,
    model_pegasus,
    tokenizer,
    column_text="dialogue",
    column_summary="summary",
    batch_size=8,        # drop to 1 if you still OOM
    max_src_len=384,     # try 256 if needed
    max_new_tokens=64,
    num_beams=1,         # keep 1 for memory; raise later if you want better quality
    use_fp16=True
)
print(score)


  0%|          | 0/103 [00:00<?, ?it/s]/tmp/ipython-input-2658500814.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (autocast_ctx() if device == "cuda" and use_fp16 else torch.no_grad()):
The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
100%|██████████| 103/103 [03:40<00:00,  2.14s/it]


{'rouge1': np.float64(0.0075544577333530386), 'rouge2': np.float64(0.00015736934011027212), 'rougeL': np.float64(0.007521433961854274), 'rougeLsum': np.float64(0.007484690196972028)}


In [19]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = {rn: score[rn] for rn in rouge_names}

pd.DataFrame([rouge_dict], index=['pegasus'])


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.007554,0.000157,0.007521,0.007485


In [20]:
# Make every dialogue a safe string (convert None / numbers / dicts, etc.)
dialogs = ["" if x is None else str(x) for x in dataset_samsum['train']['dialogue']]


In [21]:
# Count how many dialogues (this will also print the long-sequence warning if any are > model max length)
dialogue_token_len = len([tokenizer.encode(s, truncation=False) for s in dialogs])
dialogue_token_len


Token indices sequence length is longer than the specified maximum sequence length for this model (1044 > 1024). Running this sequence through the model will result in indexing errors


14732

In [43]:
MAX_SRC_LEN = 384
MAX_TGT_LEN = 256

def _to_safe_strings(seq):
    out = []
    for x in seq:
        if x is None:
            out.append("")
        elif isinstance(x, bytes):
            out.append(x.decode("utf-8", "ignore"))
        else:
            out.append(str(x))
    return out

def preprocess_fn(batch):
    # Coerce everything to strings
    inputs_text  = _to_safe_strings(batch["dialogue"])
    targets_text = _to_safe_strings(batch["summary"])

    # Tokenize inputs (no padding here)
    model_inputs = tokenizer(
        inputs_text,
        max_length=MAX_SRC_LEN,
        truncation=True,
        padding=False
    )

    # Tokenize targets correctly (no padding; collator handles it)
    labels = tokenizer(
        text_target=targets_text,
        max_length=MAX_TGT_LEN,
        truncation=True,
        padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Re-map (single process first for clearer errors)
cols = dataset_samsum["train"].column_names
dataset_samsum_pt = dataset_samsum.map(
    preprocess_fn,
    batched=True,
    remove_columns=cols,
    num_proc=None,        # keep 1 process to avoid hiding errors
    desc="Tokenizing"
)
print(dataset_samsum_pt)


Tokenizing:   0%|          | 0/14732 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/818 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})


In [23]:
dataset_samsum['train'][0]


{'id': '13818513',
 'dialogue': "Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)",
 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}

In [24]:
dataset_samsum_pt['train'][0]


{'input_ids': [12195,
  151,
  125,
  7091,
  3659,
  107,
  842,
  119,
  245,
  181,
  152,
  10508,
  151,
  7435,
  147,
  12195,
  151,
  125,
  131,
  267,
  650,
  119,
  3469,
  29344,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [12195, 7091, 3659, 111, 138, 650, 10508, 181, 3469, 107, 1]}

In [25]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_pegasus)

batch = data_collator([dataset_samsum_pt["train"][0], dataset_samsum_pt["train"][1]])
print("Non-masked target tokens:", (batch["labels"] != -100).sum().item())  # expect > 0


Non-masked target tokens: 23


In [26]:
!pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 157.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.0
    Uninstalling transformers-4.57.0:
      Successfully uninstalled transformers-4.57.0


In [29]:
from transformers import TrainingArguments  # <-- add this

training_args = TrainingArguments(
    output_dir="/content/pegasus-samsum",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=1_000_000,  # int
    report_to="none",      # avoids W&B prompt
)


In [30]:
import evaluate
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    # Convert -100 in labels back to pad_token_id for decoding
    labels = [[(token if token != -100 else tokenizer.pad_token_id) for token in seq] for seq in labels]

    # Decode to text
    preds_text = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    result = rouge.compute(predictions=preds_text, references=labels_text, use_stemmer=True)

    # Keep the common ones
    return {
        "rouge1": round(result["rouge1"] * 100, 2),
        "rouge2": round(result["rouge2"] * 100, 2),
        "rougeL": round(result["rougeL"] * 100, 2),
        "rougeLsum": round(result["rougeLsum"] * 100, 2),
    }


In [44]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

# 1) Use Seq2SeqTrainingArguments
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/pegasus-samsum",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=1_000_000,
    save_total_limit=1,
    predict_with_generate=True,
    generation_max_length=64,
    report_to="none",
    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_8bit",              # <-- All 3 flags are needed
)

# 2) Proper collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_pegasus
)

# 3) Initialize trainer
trainer = Seq2SeqTrainer(
    model=model_pegasus,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset_samsum_pt["train"],
    eval_dataset=dataset_samsum_pt["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# This should finally work
trainer.train()

Step,Training Loss
10,3.161700
20,3.213800
30,3.078100
40,3.043200
50,3.101200
60,2.967000
70,2.821200
80,2.513800
90,2.524500
100,2.421000


Step,Training Loss
10,3.161700
20,3.213800
30,3.078100
40,3.043200
50,3.101200
60,2.967000
70,2.821200
80,2.513800
90,2.524500
100,2.421000


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3922: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  UserWarning,


TrainOutput(global_step=921, training_loss=1.8401310484002908, metrics={'train_runtime': 3109.8189, 'train_samples_per_second': 4.737, 'train_steps_per_second': 0.296, 'total_flos': 5431713402470400.0, 'train_loss': 1.8401310484002908, 'epoch': 1.0})